# Persona 2 — RAG + Prompt Engineering + NVIDIA NIM

This notebook demonstrates all deliverables for **Persona 2**:

| # | Deliverable | Module |
|---|-------------|--------|
| 1 | Contextual memory (RAG) with FAISS + sentence-transformers | `persona2/rag.py` |
| 2 | Multi-persona system prompt (HypeBot, CritiBot, LurkerBot) — single LLM call → JSON | `persona2/prompts.py` |
| 3 | NVIDIA NIM (Llama 3.1) integration + JSON parsing + retry | `persona2/llm.py` |
| 4 | Artificial per-persona delays | `persona2/pipeline.py` |
| 5 | Stream-category style calibration | `persona2/prompts.py` |

In [ ]:
# Install dependencies (run once)
# !pip install faiss-cpu sentence-transformers requests numpy

In [ ]:
import os, sys, time

# Make sure persona2 package is importable
sys.path.insert(0, os.path.abspath('.'))

from persona2 import TranscriptionRAG, NIMClient, RAGLLMPipeline
from persona2.prompts import build_system_prompt, build_user_prompt, CATEGORY_STYLE

---
## 1. RAG — Contextual Memory

In [ ]:
# Initialize vector store
rag = TranscriptionRAG()
print(rag)

In [ ]:
# Simulate a gaming stream transcript arriving in chunks
transcript_chunks = [
    "Okay guys we're starting the ranked match, I'm playing Warzone today",
    "Just landed in Verdansk, going for the sniper on top of the building",
    "Oh no, there's a squad pushing from the left side already",
    "I got the sniper! Let's see if I can hit this long range shot",
    "That was a 300 meter shot, that's the longest I've ever hit!",
    "We're down to final circle, only three squads left",
]

rag.add_batch(transcript_chunks)
print(f"RAG store size: {len(rag)} chunks")

In [ ]:
# Semantic retrieval — relevant to current moment
query = "sniper long range shot"
results = rag.retrieve(query, top_k=3)

print(f"Top {len(results)} chunks most similar to: '{query}'")
for i, chunk in enumerate(results, 1):
    print(f"  [{i}] (idx={chunk.index}) {chunk.text!r}")

In [ ]:
# Recent context — last N chunks
recent = rag.get_recent_text(n=3)
print("Recent context:")
print(recent)

---
## 2. Multi-Persona Prompts

In [ ]:
# View all supported stream categories
for cat, style in CATEGORY_STYLE.items():
    print(f"\n[{cat.upper()}]")
    print(style[:120], '...' if len(style) > 120 else '')

In [ ]:
# Build the system prompt for a gaming stream
system_prompt = build_system_prompt(stream_category="gaming")
print(system_prompt)

In [ ]:
# Build the user prompt with context
latest = "That was a 300 meter shot, that's the longest I've ever hit!"

user_prompt = build_user_prompt(
    latest_message=latest,
    recent_context=rag.get_recent_text(n=3),
    semantic_context=rag.retrieve_text(latest, top_k=2),
)
print(user_prompt)

In [ ]:
# Compare prompts across categories — show how style changes
for cat in ["gaming", "just chatting", "music"]:
    sp = build_system_prompt(cat)
    # Extract just the Category Context section
    start = sp.find("## Stream Category")
    end = sp.find("## Instructions")
    print(f"\n--- {cat.upper()} ---")
    print(sp[start:end].strip())

---
## 3. NVIDIA NIM Integration

Get your free API key at https://build.nvidia.com/

In [ ]:
# Set your key — or export NVIDIA_API_KEY in your shell
API_KEY = os.environ.get("NVIDIA_API_KEY", "nvapi-YOUR-KEY-HERE")

# Initialize client
client = NIMClient(api_key=API_KEY)
print(client)

In [ ]:
# Single LLM call — returns JSON with all 3 personas
responses = client.generate(system_prompt=system_prompt, user_prompt=user_prompt)

print("Raw LLM response (parsed):")
for persona, msg in responses.items():
    print(f"  {persona:12s}: {msg}")

---
## 4. Full Pipeline — RAG + LLM + Delays

In [ ]:
# Initialize the full pipeline
pipeline = RAGLLMPipeline(
    api_key=API_KEY,
    stream_category="gaming",
)
print(pipeline)

In [ ]:
# Feed the stream history first (simulating what Persona 1 / STT provides)
pipeline.rag.add_batch(transcript_chunks[:-1])  # all but the last utterance

print(f"Pipeline RAG pre-loaded with {pipeline.rag_size} chunks")

In [ ]:
# Process the latest streamer utterance — observe per-persona delays
utterance = "That was a 300 meter shot, that's the longest I've ever hit!"

print(f"Processing: '{utterance}'")
print("-" * 60)
t0 = time.time()

messages = pipeline.process(utterance)

for msg in messages:
    elapsed = msg.timestamp - t0
    print(f"  +{elapsed:.1f}s  [{msg.persona:12s}]  {msg.text}")

In [ ]:
# Multi-turn demonstration — 3 utterances in a row
stream_moments = [
    "I'm going aggressive, pushing this last squad",
    "One down! Two more left, they're camping in the building",
    "WE WON! First win of the day, let's go!",
]

all_messages = []
for utterance in stream_moments:
    print(f"\n[STREAMER] {utterance}")
    msgs = pipeline.process(utterance)
    for msg in msgs:
        print(f"  [{msg.persona:12s}] {msg.text}")
    all_messages.extend(msgs)

print(f"\nTotal messages generated: {len(all_messages)}")
print(f"RAG store now has {pipeline.rag_size} chunks")

---
## 5. Category Switching Demo

The EDA finding: message style/length varies significantly by stream category.
The pipeline adjusts the system prompt on-the-fly when the category changes.

In [ ]:
# Just Chatting — expect longer, conversational messages
pipeline.set_category("just chatting")
print(f"Category switched to: {pipeline.stream_category}")

utterance_jc = "So I've been thinking, should I move to a bigger city or stay where I am?"
print(f"\n[STREAMER] {utterance_jc}")
msgs = pipeline.process(utterance_jc)
for msg in msgs:
    print(f"  [{msg.persona:12s}] {msg.text}  (len={len(msg.text)})")

In [ ]:
# Music — short reactive messages
pipeline.set_category("music")
utterance_music = "This beat is absolutely fire, I've been listening to this on repeat"
print(f"[STREAMER] {utterance_music}")
msgs = pipeline.process(utterance_music)
for msg in msgs:
    print(f"  [{msg.persona:12s}] {msg.text}  (len={len(msg.text)})")

---
## 6. Async Usage (for Gradio integration)

Persona 3 (Gradio UI) should call `pipeline.aprocess()` to get messages
streamed in real-time without blocking the event loop.

In [ ]:
import asyncio

pipeline.set_category("gaming")

async def demo_async():
    utterance = "Just clutched the 1v3, how is that even possible!"
    print(f"[STREAMER] {utterance}")
    async for msg in pipeline.aprocess(utterance):
        print(f"  [{msg.persona:12s}] {msg.text}")

await demo_async()

---
## Architecture Summary

```
Streamer audio (from Persona 1 / Whisper STT)
        │
        ▼
TranscriptionRAG.add(utterance)
        │
        ├─► retrieve_text(utterance)   ← semantic context (FAISS)
        └─► get_recent_text()          ← temporal context
        │
        ▼
build_system_prompt(category)   +   build_user_prompt(latest, recent, semantic)
        │
        ▼
NIMClient.generate()   →   NVIDIA NIM / Llama 3.1 API
        │
        ▼
  {"HypeBot": "...", "CritiBot": "...", "LurkerBot": "..."}
        │
        ▼
_schedule()  →  per-persona random delay
        │
        ▼
ChatMessage objects → Gradio UI (Persona 3)
```

| Component | Implementation | Key Design Choice |
|-----------|---------------|-------------------|
| Vector store | FAISS IndexFlatIP + L2 norm | Exact cosine search, no server needed |
| Embeddings | `all-MiniLM-L6-v2` | 384-dim, fast CPU inference, free |
| LLM | NVIDIA NIM Llama 3.1 8B | Free tier, single call → 3 personas |
| Prompt | System + User split | System sets personas/style, User provides live context |
| Delays | Random per-persona ranges | HypeBot: 0.5-2s, CritiBot: 2-5s, LurkerBot: 4-9s |
| Category | Injected into system prompt | Calibrates length/tone per EDA findings |